In [1]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from collections import deque

# load model & scaler
model = load_model("lstm_driving_model.keras")
scaler = joblib.load("scaler.pkl")

features = ['X_Acc','Y_Acc','Z_Acc','X_Gyro','Y_Gyro','Z_Gyro']
SEQ_LEN = 30

In [2]:
buffer = deque(maxlen=SEQ_LEN)

In [3]:
score_history = []

def smooth_score(new_score):
    score_history.append(new_score)

    if len(score_history) > 5:
        score_history.pop(0)

    return sum(score_history) / len(score_history)

In [4]:
def add_sensor_data(acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z):
    buffer.append([acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z])

In [5]:
def check_alert(risk_score):

    if risk_score >= 3:
        return "🚨 HIGH RISK! Dangerous Driving"

    elif risk_score >= 2:
        return "⚠️ Moderate Risk"

    else:
        return "✅ Safe Driving"

In [6]:
def predict_realtime():

    if len(buffer) < SEQ_LEN:
        return "Collecting data..."

    # convert to DataFrame (fix warning)
    data = pd.DataFrame(buffer, columns=features)

    # normalize
    data = scaler.transform(data)

    # safety check
    if np.std(data) < 0.01:
        return "Low variation in input data"

    # reshape
    data = np.expand_dims(data, axis=0)

    # predict
    probs = model.predict(data)[0]

    print("Probabilities:", probs)   # debug

    pred_class = int(np.argmax(probs) + 1)

    # driving score
    driving_score = float(np.sum(probs * np.array([1,2,3,4,5])))

    # 🔥 smoothing
    driving_score = smooth_score(driving_score)

    # risk score
    risk_score = float(5 - driving_score)

    # alert
    alert = check_alert(risk_score)

    return {
        "Rating": pred_class,
        "Driving Score": round(driving_score, 2),
        "Risk Score": round(risk_score, 2),
        "Confidence": float(round(np.max(probs), 2)),
        "Alert": alert
    }

In [8]:
df = pd.read_csv("merged_1star.csv")

df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.drop(columns=['Rating'], errors='ignore')

step = 0

for i, row in df.iterrows():

    add_sensor_data(
        row['X_Acc'], row['Y_Acc'], row['Z_Acc'],
        row['X_Gyro'], row['Y_Gyro'], row['Z_Gyro']
    )

    step += 1

    # predict every SEQ_LEN steps
    if step % SEQ_LEN == 0:
        result = predict_realtime()

        if isinstance(result, dict):
            print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Probabilities: [9.9998081e-01 1.6444299e-05 2.5842589e-06 1.3585185e-07 5.6686451e-09]
{'Rating': 1, 'Driving Score': 2.62, 'Risk Score': 2.38, 'Confidence': 1.0, 'Alert': '⚠️ Moderate Risk'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Probabilities: [9.9994636e-01 2.2657365e-05 3.0038094e-05 6.8682840e-07 2.5551341e-07]
{'Rating': 1, 'Driving Score': 2.22, 'Risk Score': 2.78, 'Confidence': 1.0, 'Alert': '⚠️ Moderate Risk'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Probabilities: [9.9999654e-01 2.7123187e-06 7.2066820e-07 6.0816063e-09 1.7290568e-10]
{'Rating': 1, 'Driving Score': 1.42, 'Risk Score': 3.58, 'Confidence': 1.0, 'Alert': '🚨 HIGH RISK! Dangerous Driving'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Probabilities: [9.9970239e-01 2.8405254e-04 1.2032961e-05 1.4684921e-06 4.6436611e-08]
{'Rating': 1, 'Driving Score': 1.4, 'Risk Score': 3.6, 'Confidence': 1.0, 'Alert': '🚨 HIGH RISK! Dangerous Driving'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Probabilities: [

KeyboardInterrupt: 